In [1]:
!pip install torch numpy matplotlib


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: python3.11 -m pip install --upgrade pip


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

# --- Config ---
ALPHA      = 1.0
T_MAX      = 0.01
N_DOMAIN   = 10000
N_BOUNDARY = 800
N_INITIAL  = 500
W_PDE = 1
W_BC  = 10
W_IC  = 10
T_EVAL     = [0.001, 0.005, 0.010]
LX_RANGE   = [0.5, 2.0]
LY_RANGE   = [0.5, 2.0]

# Fourier feature config — same bands as phase 2
SIGMA_BANDS = [1, 3, 5]
M_FREQ      = 32  # random frequencies per band; total Fourier features = 3×2×32 = 192

N_ADAM  = 10000
N_LBFGS = 500

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)

print(f"Device: {DEVICE}")
print(f"Fourier features: {len(SIGMA_BANDS)} bands × 2 × {M_FREQ} = {len(SIGMA_BANDS)*2*M_FREQ}")
print(f"MLP input size: {len(SIGMA_BANDS)*2*M_FREQ + 2}  (192 Fourier + 2 geometry)")
print("Config loaded.")

Device: cpu
Fourier features: 3 bands × 2 × 32 = 192
MLP input size: 194  (192 Fourier + 2 geometry)
Config loaded.


In [3]:
def T_exact(x, y, t, Lx, Ly, alpha=ALPHA):
    """
    Two-mode exact solution on [0,Lx]×[0,Ly]:
      mode 1: sin(πx/Lx)sin(πy/Ly) · exp(−λ₁t)
      mode 2: 0.3·sin(5πx/Lx)sin(5πy/Ly) · exp(−λ₂t)
    λ₁ = π²α(1/Lx² + 1/Ly²),  λ₂ = 25λ₁
    """
    lam1 = np.pi**2 * alpha * (1/Lx**2 + 1/Ly**2)
    lam2 = 25 * lam1
    low  = np.sin(np.pi * x / Lx) * np.sin(np.pi * y / Ly) * np.exp(-lam1 * t)
    high = 0.3 * np.sin(5*np.pi*x/Lx) * np.sin(5*np.pi*y/Ly) * np.exp(-lam2 * t)
    return low + high

# Sanity check 1: unit-square case matches phase 2 at t=0 (IC)
x0, y0 = np.array([0.5]), np.array([0.3])
val = T_exact(x0, y0, 0.0, 1.0, 1.0)
ic  = np.sin(np.pi*x0)*np.sin(np.pi*y0) + 0.3*np.sin(5*np.pi*x0)*np.sin(5*np.pi*y0)
assert np.isclose(val, ic).all(), f"Unit-square IC mismatch: {val} vs {ic}"
print(f"Sanity 1 passed: T_exact(0.5,0.3,0,1,1) = {val[0]:.6f}")

# Sanity check 2: BC is zero at x=0 for any Lx, Ly, t
val_bc = T_exact(np.array([0.0]), y0, 0.005, 1.5, 0.8)
assert np.isclose(val_bc, 0.0).all(), f"BC not zero: {val_bc}"
print(f"Sanity 2 passed: T_exact(0,y,t,Lx,Ly) = {val_bc[0]:.6f} (should be 0)")

# Sanity check 3: larger plate decays more slowly
lam1_small = np.pi**2 * ALPHA * (1/0.5**2 + 1/0.5**2)
lam1_large = np.pi**2 * ALPHA * (1/2.0**2 + 1/2.0**2)
assert lam1_small > lam1_large, "Small plate should decay faster"
print(f"Sanity 3 passed: λ₁(0.5×0.5)={lam1_small:.2f} > λ₁(2×2)={lam1_large:.2f}")

print("\nDecay rates at unit square (Lx=Ly=1):")
lam1 = np.pi**2 * ALPHA * 2
print(f"  λ₁ = {lam1:.4f}  (mode 1 half-life = {np.log(2)/lam1*1000:.2f} ms)")
print(f"  λ₂ = {25*lam1:.4f}  (mode 2 half-life = {np.log(2)/(25*lam1)*1000:.2f} ms)")

Sanity 1 passed: T_exact(0.5,0.3,0,1,1) = 0.509017
Sanity 2 passed: T_exact(0,y,t,Lx,Ly) = 0.000000 (should be 0)
Sanity 3 passed: λ₁(0.5×0.5)=78.96 > λ₁(2×2)=4.93

Decay rates at unit square (Lx=Ly=1):
  λ₁ = 19.7392  (mode 1 half-life = 35.12 ms)
  λ₂ = 493.4802  (mode 2 half-life = 1.40 ms)
